# Day 51 — A byte-level BPE tokenizer from scratch

A language model never sees your text. It sees a list of integers, and the thing that
produced that list made a long series of choices you did not make: where words break,
whether a leading space belongs to the next word, how a number is cut up, and how many
of those integers your sentence costs.

This notebook builds the whole tokenizer — train, encode, decode — in about a hundred
lines, and then measures the four places it quietly decides something on your behalf.

In [1]:
import re
from collections import Counter

# A fixed corpus, so every number on this page is reproducible.  Deliberately mixed:
# prose, numbers, code-ish punctuation and a line of Chinese.
CORPUS = """
the cost of a model call is counted in tokens, not in words and not in characters.
a token is whatever the tokenizer decided a token is, and the tokenizer was trained
on a corpus you never saw. the same sentence costs a different number of tokens in
every model, because every model merged a different set of pairs in a different order.
tokenization is the first layer of the model and the only one that is not learned by
gradient descent. it is learned by counting.
the training loop is three lines: count every adjacent pair, merge the most frequent
one, repeat. the merges are kept in the order they were made, and encoding replays
them in that order. decoding is the reverse: look up the bytes of every token and
join them back together.
a tokenizer that cannot reproduce its input is broken, however good its compression
is. the test is a round trip: encode, decode, compare bytes.
counting words is not counting tokens. counting characters is not counting tokens.
the only thing that counts tokens is the tokenizer.
prices are quoted per million tokens: 0.15, 0.60, 1.25, 10.00 per million.
a context window of 128000 tokens is not 128000 words.
def encode(text): return [vocab[b] for b in text.encode()]
模型看到的從來不是你的文字，而是一串整數。
"""


# ================================================================ 1. bytes
def to_bytes(text):
    """Text becomes bytes before it becomes anything else.

    Working in bytes rather than characters is what makes a tokenizer total: there
    are only 256 possible bytes, every one of them is in the vocabulary from the
    start, and no input can produce an unknown token.  The cost is that one Chinese
    character is three bytes and one emoji can be four.
    """
    return list(text.encode('utf-8'))


# ================================================================ 2. pre-tokenising
# The pattern decides what BPE is even allowed to merge.  Two rules carry most of the
# weight: a leading space is glued to the word that follows it, and a run of digits is
# cut into groups of at most three.
PRETOKEN = re.compile(r"""'(?:[sdmt]|ll|ve|re)| ?[^\W\d_]+| ?\d{1,3}| ?[^\s\w]+|\s+""")


def pretokenize(text, pattern=None):
    """Split text into chunks that BPE is allowed to merge inside, never across.

    Without this step the most frequent pair in English prose is (space, t), and the
    tokenizer spends its whole budget learning fragments that straddle word
    boundaries.  With it, ' the' is one chunk and can become one token.
    """
    return [m.group(0) for m in (pattern or PRETOKEN).finditer(text)]


print(f'corpus: {len(CORPUS)} characters, {len(CORPUS.encode("utf-8"))} bytes')
print('first chunks:', pretokenize(CORPUS)[:12])
print()
for t in ('a', 'é', '模', '🙂'):
    print(f'{t!r:>6}  {len(t.encode("utf-8"))} byte(s)  {to_bytes(t)}')

corpus: 1234 characters, 1276 bytes
first chunks: ['\n', 'the', ' cost', ' of', ' a', ' model', ' call', ' is', ' counted', ' in', ' tokens', ',']

   'a'  1 byte(s)  [97]
   'é'  2 byte(s)  [195, 169]
   '模'  3 byte(s)  [230, 168, 161]
   '🙂'  4 byte(s)  [240, 159, 153, 130]


## Training: count every adjacent pair, merge the most frequent, repeat

The merges are kept **in the order they were made**. That order is the tokenizer —
encoding is nothing but replaying it.

Ties are broken by taking the lowest pair of ids, so two runs on the same corpus give
the same vocabulary. Without a written-down rule, `max` follows dictionary order.

In [2]:
# ================================================================ 3. training
def pair_counts(words):
    """How often each adjacent pair occurs, summed over the corpus.

    `words` maps a tuple of token ids to how many times that chunk appeared, so a
    chunk that occurs 400 times contributes 400 to each of its pairs - the counting
    is over occurrences in the corpus, not over distinct chunks.
    """
    counts = Counter()
    for word, freq in words.items():
        for a, b in zip(word, word[1:]):
            counts[(a, b)] += freq
    return counts


def merge_word(word, pair, new_id):
    """Replace every occurrence of `pair` in one chunk, left to right, without overlap."""
    out, i = [], 0
    while i < len(word):
        if i + 1 < len(word) and (word[i], word[i + 1]) == pair:
            out.append(new_id)
            i += 2
        else:
            out.append(word[i])
            i += 1
    return tuple(out)


def train(text, vocab_size, verbose_until=0, pattern=None):
    """Count pairs, merge the most frequent, repeat until the vocabulary is full.

    Returns (merges, vocab).  `merges` is an ordered list of (pair -> new id); the
    order is part of the tokenizer, not an implementation detail - encoding replays
    exactly this order.  `vocab` maps every id to the bytes it stands for.

    Ties are broken by taking the lowest pair of ids, so two runs on the same corpus
    produce the same tokenizer.  Without that rule `max` would follow dictionary
    order and the vocabulary would change between Python versions.
    """
    words = Counter(tuple(to_bytes(chunk)) for chunk in pretokenize(text, pattern))
    vocab = {i: bytes([i]) for i in range(256)}
    merges, trace = [], []
    while len(vocab) < vocab_size:
        counts = pair_counts(words)
        if not counts:
            break
        best = max(counts.items(), key=lambda kv: (kv[1], -kv[0][0], -kv[0][1]))[0]
        if counts[best] < 2:
            break                      # nothing repeats any more; merging would not compress
        new_id = len(vocab)
        vocab[new_id] = vocab[best[0]] + vocab[best[1]]
        words = {merge_word(w, best, new_id): f for w, f in words.items()}
        merges.append((best, new_id))
        if len(merges) <= verbose_until:
            trace.append((len(merges), best, counts[best], vocab[new_id]))
    return merges, vocab, trace


merges, vocab, trace = train(CORPUS, 400, verbose_until=8)
print(f'{len(merges)} merges, vocabulary {len(vocab)}')
print()
print(f'{"#":>3} {"pair":>12} {"count":>7}   new token')
for n, pair, cnt, piece in trace:
    print(f'{n:>3} {str(pair):>12} {cnt:>7}   {piece.decode("utf-8", "replace")!r}')

143 merges, vocabulary 399

  #         pair   count   new token
  1    (32, 116)      41   ' t'
  2   (101, 110)      29   'en'
  3   (101, 114)      29   'er'
  4    (32, 105)      26   ' i'
  5   (256, 104)      21   ' th'
  6    (99, 111)      20   'co'
  7   (111, 107)      17   'ok'
  8   (262, 257)      16   'oken'


## The same word, two different prices

A leading space is not whitespace to a tokenizer. It is part of the word, because that
is how the word appears in running text.

In [3]:
# ================================================================ 4. encode / decode
def encode(text, merges, pattern=None):
    """Apply the merges in the order they were learned - not longest-match.

    Longest-match would be a different algorithm with a different output; BPE is
    defined by the replay, which is why the merge list has to ship with the model.
    """
    rank = {pair: i for i, (pair, _) in enumerate(merges)}
    new_of = {pair: nid for pair, nid in merges}
    out = []
    for chunk in pretokenize(text, pattern):
        ids = tuple(to_bytes(chunk))
        while len(ids) >= 2:
            pairs = set(zip(ids, ids[1:]))
            candidates = [p for p in pairs if p in rank]
            if not candidates:
                break
            best = min(candidates, key=lambda p: rank[p])
            ids = merge_word(ids, best, new_of[best])
        out.extend(ids)
    return out


def decode(ids, vocab, errors='strict'):
    """Concatenate the bytes of every token, then decode once as UTF-8.

    Decoding per token would fail on any multi-byte character that got split across
    two tokens - which is normal and fine, as long as you join first.
    """
    return b''.join(vocab[i] for i in ids).decode('utf-8', errors=errors)


def round_trips(text, merges, vocab):
    return decode(encode(text, merges), vocab) == text


for w in (' tokens', 'tokens', ' the', 'the'):
    ids = encode(w, merges)
    print(f'{w!r:>10} -> {ids}   {len(ids)} token(s)')
print()
s = 'the cost of a model call is counted in tokens'
ids = encode(s, merges)
print(f'"{s}"')
print(f'{len(s.split())} words, {len(s)} characters, {len(s.encode("utf-8"))} bytes, {len(ids)} tokens')
print(' | '.join(vocab[i].decode('utf-8', 'replace') for i in ids))

 ' tokens' -> [286]   1 token(s)
  'tokens' -> [116, 263, 115]   3 token(s)
    ' the' -> [266]   1 token(s)
     'the' -> [318]   1 token(s)

"the cost of a model call is counted in tokens"
10 words, 45 characters, 45 bytes, 12 tokens
the |  cost |  of |  a |  model |  ca | ll |  is |  count | ed |  in |  tokens


## Encoding replays the merges in rank order, not longest-match

At every step it applies the merge with the **smallest rank still available** — the
earliest one learned. Longest-match over the same vocabulary is a different algorithm
with different output, which is why the merge list has to ship with the model.

In [4]:
word = ' counted'
ids_t = tuple(to_bytes(word))
rank = {pair: i for i, (pair, _) in enumerate(merges)}
new_of = {pair: nid for pair, nid in merges}
print(f'{word!r} starts as {len(ids_t)} bytes')
while len(ids_t) >= 2:
    cand = [p for p in set(zip(ids_t, ids_t[1:])) if p in rank]
    if not cand:
        break
    best = min(cand, key=lambda p: rank[p])
    ids_t = merge_word(ids_t, best, new_of[best])
    pieces = ' '.join(repr(vocab[i].decode('utf-8', 'replace')) for i in ids_t)
    print(f'after merge #{rank[best] + 1:>3}:  {pieces}')

' counted' starts as 8 bytes
after merge #  6:  ' ' 'co' 'u' 'n' 't' 'e' 'd'
after merge # 15:  ' co' 'u' 'n' 't' 'e' 'd'
after merge # 19:  ' co' 'u' 'nt' 'e' 'd'
after merge # 25:  ' co' 'unt' 'e' 'd'
after merge # 32:  ' count' 'e' 'd'
after merge # 34:  ' count' 'ed'


## Silent failure #1: the most frequent pair is usually not unique

Every tie is a fork. Other tie-breaks give 141 to 143 merges and 500 to 504 tokens on
this corpus, so the choices compress to within a few tokens of each other and disagree
about what a token is - which makes the tie-break part of the specification — and picking the
k most frequent things with a written-down tie-break is LeetCode 692, below.

In [5]:
def tie_report(text, vocab_size):
    """How often the most frequent pair is not unique.

    Every tie is a fork: two tokenizers that compress the same corpus to within a few
    tokens of each other and disagree about what a token is.  The tie-break has to be
    written down.
    """
    words = Counter(tuple(to_bytes(c)) for c in pretokenize(text))
    vocab = {i: bytes([i]) for i in range(256)}
    ties, steps, first = 0, 0, None
    while len(vocab) < vocab_size:
        counts = pair_counts(words)
        if not counts:
            break
        top = max(counts.values())
        if top < 2:
            break
        tied = [p for p, c in counts.items() if c == top]
        steps += 1
        if len(tied) > 1:
            ties += 1
            if first is None:
                first = (steps, top, sorted(tied)[:3], len(tied))
        best = max(counts.items(), key=lambda kv: (kv[1], -kv[0][0], -kv[0][1]))[0]
        nid = len(vocab)
        vocab[nid] = vocab[best[0]] + vocab[best[1]]
        words = {merge_word(w, best, nid): f for w, f in words.items()}
    return steps, ties, first


steps, ties, first = tie_report(CORPUS, 400)
print(f'{ties} of {steps} merge steps ({100 * ties / steps:.0f}%) had a tie at the top')
st, cnt, examples, howmany = first
shown = ', '.join(repr(vocab[a].decode('utf-8', 'replace') + vocab[b].decode('utf-8', 'replace'))
                  for a, b in examples)
print(f'the first is at step {st}: {howmany} pairs all occur {cnt} times - {shown}')

123 of 143 merge steps (86%) had a tie at the top
the first is at step 2: 2 pairs all occur 29 times - 'en', 'er'


## Silent failure #2: decoding one token at a time

A multi-byte character gets split across token boundaries all the time, and that is
fine — as long as you join the bytes first and decode **once**. The round trip is the
property most worth testing, and `check()` asserts five more besides.
`errors='replace'` is what turns a boundary bug into mojibake instead of an exception.

In [6]:
def _valid_utf8(b):
    """Is this run of bytes a complete UTF-8 string by itself?"""
    try:
        b.decode('utf-8')
        return True
    except UnicodeDecodeError:
        return False


zh = '模型看到的'
ids = encode(zh, merges)
bad = sum(1 for i in ids if not _valid_utf8(vocab[i]))
print(f'{zh!r} -> {len(ids)} tokens, {bad} of them not valid UTF-8 on their own')
print('join then decode  :', repr(decode(ids, vocab)))
print('decode then join  :', repr(''.join(vocab[i].decode('utf-8', 'replace') for i in ids)))
print()
print('round trip over the whole corpus:', round_trips(CORPUS, merges, vocab))

'模型看到的' -> 13 tokens, 12 of them not valid UTF-8 on their own
join then decode  : '模型看到的'
decode then join  : '������������的'

round trip over the whole corpus: True


## Silent failure #3: the regex decides what a token may be

Pre-tokenisation cuts the text into chunks that merges may never cross. Dropping it
barely changes the compression — what it changes is *what a token is*.

In [7]:
whole = re.compile(r'[\s\S]+')               # the whole text as one chunk
m_no, v_no, _ = train(CORPUS, 400, pattern=whole)
n_no = len(encode(CORPUS, m_no, pattern=whole))
cross_no = sum(1 for _, nid in m_no if b' ' in v_no[nid][1:-1])
n_yes = len(encode(CORPUS, merges))
cross_yes = sum(1 for _, nid in merges if b' ' in vocab[nid][1:-1])
nbytes = len(CORPUS.encode('utf-8'))
print(f'{"":<22}{"merges":>8}{"tokens":>8}{"tokens/byte":>13}{"spaced tokens":>15}')
print(f'{"one chunk, no regex":<22}{len(m_no):>8}{n_no:>8}{n_no / nbytes:>13.4f}{cross_no:>15}')
print(f'{"pre-tokenised":<22}{len(merges):>8}{n_yes:>8}{n_yes / nbytes:>13.4f}{cross_yes:>15}')
print()
print('tokens with a space inside:',
      [v_no[nid].decode('utf-8', 'replace') for _, nid in m_no if b' ' in v_no[nid][1:-1]][:4])

                        merges  tokens  tokens/byte  spaced tokens
one chunk, no regex        144     502       0.3934             12
pre-tokenised              143     500       0.3918              0

tokens with a space inside: [' is not ', 't in', 't is ', ' the tokeniz']


## What the vocabulary size buys you, and how it differs from Day 38

Huffman merges the two **least** frequent symbols to shorten the codes for a fixed
alphabet. BPE merges the **most** frequent pair to shorten the sequence, and is allowed
to invent new symbols. Same greedy shape, opposite end of the frequency table.

In [8]:
# ================================================================ 5. measurements
def tokens_per_byte(text, merges, vocab, pattern=None):
    ids = encode(text, merges, pattern)
    return len(ids), len(text.encode('utf-8')), len(ids) / len(text.encode('utf-8'))


def huffman_bits_per_byte(data):
    """Day 38's Huffman, for contrast: merge the two LEAST frequent symbols.

    Huffman keeps the alphabet and shortens the codes.  BPE keeps the code length and
    shortens the alphabet - it merges the MOST frequent pair into a new symbol.  Same
    greedy shape, opposite end of the frequency table, different unit of saving.
    """
    import heapq
    freq = Counter(data)
    if len(freq) == 1:
        return 1.0
    heap = [(f, i, None) for i, (s, f) in enumerate(freq.items())]
    lengths = {s: 0 for s in freq}
    groups = {i: [s] for i, (s, f) in enumerate(freq.items())}
    heapq.heapify(heap)
    nxt = len(freq)
    while len(heap) > 1:
        f1, i1, _ = heapq.heappop(heap)
        f2, i2, _ = heapq.heappop(heap)
        for s in groups[i1] + groups[i2]:
            lengths[s] += 1
        groups[nxt] = groups[i1] + groups[i2]
        heapq.heappush(heap, (f1 + f2, nxt, None))
        nxt += 1
    total = sum(freq.values())
    return sum(freq[s] * lengths[s] for s in freq) / total


print(f'{"vocab":>7}{"merges":>8}{"tokens":>8}{"tokens/byte":>13}')
for size in (256, 300, 350, 400):
    mm, vv, _ = train(CORPUS, size)
    n, b, r = tokens_per_byte(CORPUS, mm, vv)
    print(f'{len(vv):>7}{len(mm):>8}{n:>8}{r:>13.4f}')
print()
bits = huffman_bits_per_byte(CORPUS.encode('utf-8'))
n, b, r = tokens_per_byte(CORPUS, merges, vocab)
print(f'Huffman: {bits:.4f} bits per byte (alphabet unchanged, codes shorter)')
print(f'BPE:     {r:.4f} tokens per byte (sequence shorter, alphabet grew to {len(vocab)})')

  vocab  merges  tokens  tokens/byte
    256       0    1276       1.0000
    300      44     767       0.6011
    350      94     598       0.4687
    399     143     500       0.3918

Huffman: 4.6928 bits per byte (alphabet unchanged, codes shorter)
BPE:     0.3918 tokens per byte (sequence shorter, alphabet grew to 399)


## LeetCode 692 — Top K Frequent Words

Return the k most frequent words, most frequent first, ties broken alphabetically. That
second clause is the BPE merge loop's tie-break, written as an interview question.

In [9]:
# ================================================================ 6. LeetCode 692
class SolutionTopKFrequent:
    """LeetCode 692 - Top K Frequent Words.

    Return the k most frequent words, most frequent first, and break ties by
    alphabetical order.  That second clause is the whole problem, and it is the same
    clause a BPE trainer needs: without a written-down tie-break, two runs that
    compress to within a few tokens of each other disagree about what a token is.
    """

    def topKFrequent(self, words: list[str], k: int) -> list[str]:
        count = Counter(words)
        # sort by falling frequency, then rising word - one key, both rules
        return sorted(count, key=lambda w: (-count[w], w))[:k]


class _RevStr:
    """A string that sorts backwards, so a min-heap evicts the alphabetically last."""

    __slots__ = ('s',)

    def __init__(self, s):
        self.s = s

    def __lt__(self, other):
        return self.s > other.s

    def __eq__(self, other):
        return self.s == other.s


def top_k_frequent_heap(words, k):
    """The same answer with a heap of size k, for when the vocabulary is large.

    The comparison has to be inverted on both fields at once, which is why a tuple of
    (count, reversed-string) does not work and a wrapper class or negated count does.
    """
    import heapq
    count = Counter(words)
    heap = []
    for w, c in count.items():
        heapq.heappush(heap, (c, _RevStr(w)))
        if len(heap) > k:
            heapq.heappop(heap)
    return [w.s for _, w in sorted(heap, key=lambda t: (-t[0], t[1].s))]


ex1 = ['i', 'love', 'leetcode', 'i', 'love', 'coding']
ex2 = ['the', 'day', 'is', 'sunny', 'the', 'the', 'the', 'sunny', 'is', 'is']
print(ex1, '-> k=2 ->', SolutionTopKFrequent().topKFrequent(ex1, 2))
print(ex2, '-> k=4 ->', SolutionTopKFrequent().topKFrequent(ex2, 4))
print('heap version:', top_k_frequent_heap(ex1, 2))

['i', 'love', 'leetcode', 'i', 'love', 'coding'] -> k=2 -> ['i', 'love']
['the', 'day', 'is', 'sunny', 'the', 'the', 'the', 'sunny', 'is', 'is'] -> k=4 -> ['the', 'is', 'sunny', 'day']
heap version: ['i', 'love']


## LeetCode 394 — Decode String

One stack, two things pushed at every opening bracket: the text built so far and the
repeat count waiting for the text still to come. Nothing expands until the closing
bracket arrives.

In [10]:
# ================================================================ 7. LeetCode 394
class SolutionDecodeString:
    """LeetCode 394 - Decode String.

    Expand `3[a2[c]]` into `accaccacc`.  One stack, two things pushed per bracket:
    the text built so far and the repeat count waiting for it.  A tokenizer's decode
    step is the same move without nesting - look each id up and concatenate - and
    both are wrong in the same way if you expand eagerly instead of on the closing
    bracket.
    """

    def decodeString(self, s: str) -> str:
        stack, cur, num = [], '', 0
        for ch in s:
            if ch.isdigit():
                num = num * 10 + int(ch)       # digits can be multi-character
            elif ch == '[':
                stack.append((cur, num))
                cur, num = '', 0
            elif ch == ']':
                prev, times = stack.pop()
                cur = prev + cur * times
            else:
                cur += ch
        return cur


def decode_string_brute(s):
    """Repeatedly expand the innermost bracket, for checking the stack version."""
    pat = re.compile(r'(\d+)\[([a-zA-Z]*)\]')
    while '[' in s:
        s, n = pat.subn(lambda m: m.group(2) * int(m.group(1)), s)
        if n == 0:
            break
    return s


for s_ in ('3[a]2[bc]', '3[a2[c]]', '2[abc]3[cd]ef', '12[a]'):
    print(f'{s_!r:>16} -> {SolutionDecodeString().decodeString(s_)!r}')

     '3[a]2[bc]' -> 'aaabcbc'
      '3[a2[c]]' -> 'accaccacc'
 '2[abc]3[cd]ef' -> 'abcabccdcdcdef'
         '12[a]' -> 'aaaaaaaaaaaa'


## Tests

Everything above, re-derived from scratch and asserted.

In [11]:
# ================================================================ 8. assertions
def check():
    merges, vocab, _ = train(CORPUS, 400)

    # the round trip is the property most worth testing - five more follow
    for s in [CORPUS, 'the cost of a model call', '模型看到的從來不是你的文字', '',
              'a', ' ', '\n\n', '0.15 per million', 'def f(x): return x ** 2',
              'ÄÖÜ äöü ß — – … “quoted”', '🙂🙃', 'x' * 500]:
        assert round_trips(s, merges, vocab), repr(s[:40])

    # every token stands for a non-empty string of bytes, and ids are contiguous
    assert sorted(vocab) == list(range(len(vocab)))
    assert all(len(v) >= 1 for v in vocab.values())
    assert all(vocab[i] == bytes([i]) for i in range(256))
    # a merged token is exactly the concatenation of the two it came from
    for (a, b), nid in merges:
        assert vocab[nid] == vocab[a] + vocab[b]

    # training is deterministic: same corpus in, same merge list out
    m2, v2, _ = train(CORPUS, 400)
    assert [p for p, _ in merges] == [p for p, _ in m2] and v2 == vocab

    # merging never makes the sequence longer, and a bigger vocabulary never makes it
    # longer either
    prev = None
    for size in (256, 300, 350, 400):
        mm, vv, _ = train(CORPUS, size)
        n = len(encode(CORPUS, mm))
        assert prev is None or n <= prev
        prev = n
    assert len(encode(CORPUS, [])) == len(CORPUS.encode('utf-8'))

    # pre-tokenising never lets a merge cross a chunk boundary
    for chunk in pretokenize(CORPUS):
        assert decode(encode(chunk, merges), vocab) == chunk
    assert ''.join(pretokenize(CORPUS)) == CORPUS      # the split loses nothing

    # LeetCode 692, against a brute-force sort, with ties everywhere
    import random
    S692 = SolutionTopKFrequent()
    for seed in range(300):
        r = random.Random(seed)
        ws = [r.choice('abcde') * r.randint(1, 2) for _ in range(r.randint(1, 20))]
        k = r.randint(1, len(set(ws)))
        want = sorted(Counter(ws), key=lambda w: (-Counter(ws)[w], w))[:k]
        assert S692.topKFrequent(ws, k) == want
        assert top_k_frequent_heap(ws, k) == want
    assert S692.topKFrequent(['i', 'love', 'leetcode', 'i', 'love', 'coding'], 2) == ['i', 'love']
    assert S692.topKFrequent(
        ['the', 'day', 'is', 'sunny', 'the', 'the', 'the', 'sunny', 'is', 'is'], 4) \
        == ['the', 'is', 'sunny', 'day']

    # LeetCode 394, against repeated innermost expansion
    S394 = SolutionDecodeString()
    for s in ['3[a]2[bc]', '3[a2[c]]', '2[abc]3[cd]ef', 'abc3[cd]xyz', '10[a]',
              '2[2[2[a]]]', 'leet2[code]3[tnt]', 'a', '']:
        assert S394.decodeString(s) == decode_string_brute(s), s
    for seed in range(300):
        r = random.Random(seed + 7)
        s = ''
        for _ in range(r.randint(0, 3)):
            s += r.choice(['ab', str(r.randint(1, 12)) + '[' + 'xy' * r.randint(0, 2) + ']'])
        assert S394.decodeString(s) == decode_string_brute(s), s
    assert S394.decodeString('12[a]') == 'a' * 12        # multi-digit counts
    return True


print('check() ->', check())

check() -> True
